# Base Model Evaluation: Phi-4-mini-instruct (no fine-tuning)

**Purpose:** Establish a performance baseline using the raw, un-fine-tuned `microsoft/Phi-4-mini-instruct` model from HuggingFace. 
This baseline is compared against the fine-tuned model and the fine-tuned + RAG system to quantify how much each improvement stage contributes to Atlas's customer support quality.

**Scope:** 40 test cases — 25 policy questions + 15 behavioral/compliance scenarios.  
Operational cases (which require live DB lookups) are excluded since the base model has no tool-calling capability.

**Environment:** Google Colab with A100 GPU recommended.  
Model is loaded in 4-bit quantization (BitsAndBytes NF4) to fit within GPU memory.

**Metrics evaluated (same as compare_ft_vs_rag.py):**
1. **Policy Compliance** — empathy, next-step, length adequacy, professional tone
2. **Policy Specificity** — concrete numbers/dates/named policies in replies
3. **Response Relevance** — keyword overlap with expected answer elements
4. **Hallucination Rate** — fabricated order IDs, tracking numbers, or dollar amounts
5. **FCR Classification** — resolved / escalated / needs_followup

In [ ]:
# Cell 2: Install dependencies
# NOTE: Restart runtime after this cell completes if running on a fresh Colab instance.
!pip install -q transformers bitsandbytes accelerate torch

In [ ]:
# Cell 3: Imports and metric functions
# These five metric functions are identical to those used in compare_ft_vs_rag.py
# to ensure apples-to-apples comparison across all three model configurations.

import re
import json
import time
import numpy as np
from pathlib import Path
from statistics import mean, median

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# ── Metric 1: Policy Compliance ──────────────────────────────────────────────
# Checks four behavioral dimensions: empathy, actionable next step,
# sufficient response length, and professional tone.

def policy_compliance(reply):
    has_empathy = bool(re.search(
        r'\b(sorry|apologize|understand|frustrat|concern|glad|happy|here to help|care|appreciate)\b',
        reply, re.IGNORECASE
    ))
    has_next_step = bool(re.search(
        r"\b(I'll|I will|let me|please|here are|you can|contact|visit|initiate|follow|step|proceed)\b",
        reply, re.IGNORECASE
    ))
    adequate_length = len(reply.split()) >= 15
    bad_phrases = ["cannot help", "no idea", "i don't know", "unable to assist", "not able to help"]
    professional_tone = not any(p in reply.lower() for p in bad_phrases)
    score = (has_empathy + has_next_step + adequate_length + professional_tone) / 4
    return {"compliance_score": round(score, 2), "has_empathy": has_empathy,
            "has_next_step": has_next_step, "adequate_length": adequate_length,
            "professional_tone": professional_tone}


# ── Metric 2: Policy Specificity ─────────────────────────────────────────────
# Measures whether the reply contains concrete policy details (numbers, dates,
# named programs). This is the primary metric where RAG is expected to win.

def policy_specificity(reply, is_policy_question):
    if not is_policy_question:
        return None
    specific_patterns = [
        r'\b\d+\s*(days?|hours?|weeks?|months?|business days?)\b',
        r'\b\d+\s*%\b',
        r'\$\d+',
        r'\b(30|60|90|31)\s*days?\b',
        r'\b3[\-\u2013]5\s*business\s*days?\b',
        r'\b48\s*hours?\b',
        r'\b(january|november|december)\b',
        r'\bA[\-\u2013]to[\-\u2013]z\b',
        r'\bpre[\-\u2013]paid\s*return\s*label\b',
        r'\boriginal\s*payment\s*method\b',
    ]
    hits = sum(1 for p in specific_patterns if re.search(p, reply, re.IGNORECASE))
    return round(min(hits / 3, 1.0), 2)


# ── Metric 3: Response Relevance ─────────────────────────────────────────────
# Fraction of expected keywords that appear in the reply.

def response_relevance(reply, expected_keywords):
    if not expected_keywords:
        return 1.0
    hits = sum(1 for kw in expected_keywords if kw.lower() in reply.lower())
    return round(hits / len(expected_keywords), 2)


# ── Metric 4: Hallucination Check ────────────────────────────────────────────
# Detects fabricated order IDs, tracking numbers, or dollar amounts that
# were never present in the user's message.

def hallucination_check(reply, message):
    order_in_input = set(re.findall(r'ORD-[\w]+', message, re.IGNORECASE))
    order_in_reply = set(re.findall(r'ORD-[\w]+', reply, re.IGNORECASE))
    if order_in_reply - order_in_input:
        return 1
    track_in_input = bool(re.search(r'1Z[A-Z0-9]{16}', message))
    track_in_reply = bool(re.search(r'1Z[A-Z0-9]{16}', reply))
    if track_in_reply and not track_in_input:
        return 1
    amounts_input = set(re.findall(r'\$[\d,.]+', message))
    amounts_reply = set(re.findall(r'\$[\d,.]+', reply))
    if amounts_reply - amounts_input:
        return 1
    return 0


# ── Metric 5: FCR Classification ─────────────────────────────────────────────
# Classifies whether the reply resolves the issue, escalates it, or leaves
# it in a follow-up-needed state.

def fcr_classify(reply):
    r = reply.lower()
    if any(w in r for w in ['human agent', 'transfer', 'escalate', 'manager', 'supervisor', 'connecting you']):
        return 'escalated'
    resolution_words = ["i've", "i have", "i will", "processed", "initiated", "submitted",
                        "here are the steps", "your order", "your refund", "your account"]
    if any(w in r for w in resolution_words):
        return 'resolved'
    if reply.count('?') > 1:
        return 'needs_followup'
    return 'resolved'


print("Imports and metric functions loaded successfully.")

In [ ]:
# Cell 4: Test cases
# 40 cases: 25 policy + 15 behavioral.
# Operational cases requiring live DB access are excluded (not applicable to base model).

TEST_CASES = [
    # ── POLICY QUESTIONS (25) ─────────────────────────────────────────────────
    {"id": "POL-01", "category": "policy", "message": "What is your return policy?",
     "expected_keywords": ["return", "days"], "policy_question": True},
    {"id": "POL-02", "category": "policy", "message": "How many days do I have to return an item?",
     "expected_keywords": ["30", "days"], "policy_question": True},
    {"id": "POL-03", "category": "policy",
     "message": "I bought something as a gift for my mom. Can she return it if she doesn't like it?",
     "expected_keywords": ["return"], "policy_question": True},
    {"id": "POL-04", "category": "policy", "message": "How do I start a return for something I bought?",
     "expected_keywords": ["return", "order"], "policy_question": True},
    {"id": "POL-05", "category": "policy", "message": "Are there any items that cannot be returned?",
     "expected_keywords": ["return"], "policy_question": True},
    {"id": "POL-06", "category": "policy",
     "message": "How long does a refund take after I send the item back?",
     "expected_keywords": ["business days", "refund"], "policy_question": True},
    {"id": "POL-07", "category": "policy",
     "message": "I bought a Christmas gift in November. Can I return it in January?",
     "expected_keywords": ["January", "return"], "policy_question": True},
    {"id": "POL-08", "category": "policy",
     "message": "I missed the return window by a couple of days. Is there any exception?",
     "expected_keywords": ["return", "days"], "policy_question": True},
    {"id": "POL-09", "category": "policy",
     "message": "Do I have to pay for return shipping or is it free?",
     "expected_keywords": ["shipping", "return"], "policy_question": True},
    {"id": "POL-10", "category": "policy",
     "message": "My headphones broke after 2 months of normal use. Are they covered under warranty?",
     "expected_keywords": ["warranty"], "policy_question": True},
    {"id": "POL-11", "category": "policy",
     "message": "How do I make a warranty claim for a defective product?",
     "expected_keywords": ["warranty", "return"], "policy_question": True},
    {"id": "POL-12", "category": "policy",
     "message": "Can I exchange my item for a different size instead of getting a refund?",
     "expected_keywords": ["exchange", "return"], "policy_question": True},
    {"id": "POL-13", "category": "policy",
     "message": "What does the A-to-z Guarantee cover and how do I use it?",
     "expected_keywords": ["A-to-z", "guarantee"], "policy_question": True},
    {"id": "POL-14", "category": "policy",
     "message": "How long does standard shipping usually take?",
     "expected_keywords": ["shipping", "days"], "policy_question": True},
    {"id": "POL-15", "category": "policy",
     "message": "My package was left on my doorstep and it went missing. What happens now?",
     "expected_keywords": ["lost", "refund"], "policy_question": True},
    {"id": "POL-16", "category": "policy",
     "message": "Can I return a digital product or software download?",
     "expected_keywords": ["digital", "return"], "policy_question": True},
    {"id": "POL-17", "category": "policy",
     "message": "Will my refund go back to my credit card or will I get store credit?",
     "expected_keywords": ["refund", "payment", "original"], "policy_question": True},
    {"id": "POL-18", "category": "policy",
     "message": "I bought from a third-party seller. Can I still return directly to you?",
     "expected_keywords": ["seller", "return"], "policy_question": True},
    {"id": "POL-19", "category": "policy",
     "message": "What is a partial refund and when would I get one instead of a full refund?",
     "expected_keywords": ["partial", "refund"], "policy_question": True},
    {"id": "POL-20", "category": "policy",
     "message": "Is there a restocking fee if I return an item without the original box?",
     "expected_keywords": ["restocking", "return"], "policy_question": True},
    {"id": "POL-21", "category": "policy",
     "message": "What is the return window specifically for electronics?",
     "expected_keywords": ["return", "days"], "policy_question": True},
    {"id": "POL-22", "category": "policy",
     "message": "If the seller does not respond to my return request, what are my options?",
     "expected_keywords": ["seller", "A-to-z"], "policy_question": True},
    {"id": "POL-23", "category": "policy",
     "message": "Can I get my refund as gift card balance instead of back to my bank?",
     "expected_keywords": ["gift card", "refund"], "policy_question": True},
    {"id": "POL-24", "category": "policy",
     "message": "How long after approval does it take for the refund to show on my bank statement?",
     "expected_keywords": ["business days", "bank", "refund"], "policy_question": True},
    {"id": "POL-25", "category": "policy",
     "message": "I returned an item but my refund was denied. Why would that happen?",
     "expected_keywords": ["denied", "return", "refund"], "policy_question": True},

    # ── BEHAVIORAL / COMPLIANCE CASES (15) ────────────────────────────────────
    {"id": "BEH-01", "category": "behavioral",
     "message": "I'm very frustrated. My package has been delayed for 2 weeks and nobody is helping me.",
     "expected_keywords": ["sorry", "understand", "help"], "policy_question": False},
    {"id": "BEH-02", "category": "behavioral",
     "message": "This is the third problem I've had with your service. I want to speak to a manager.",
     "expected_keywords": ["sorry", "escalate", "agent"], "policy_question": False},
    {"id": "BEH-03", "category": "behavioral",
     "message": "I need help right now, it is urgent.",
     "expected_keywords": ["help", "assist"], "policy_question": False},
    {"id": "BEH-04", "category": "behavioral",
     "message": "Thank you so much, you've been really helpful!",
     "expected_keywords": ["welcome", "glad", "help"], "policy_question": False},
    {"id": "BEH-05", "category": "behavioral",
     "message": "I'm not happy with my purchase at all.",
     "expected_keywords": ["sorry", "help", "return"], "policy_question": False},
    {"id": "BEH-06", "category": "behavioral",
     "message": "Your website is not working and I cannot place my order.",
     "expected_keywords": ["sorry", "help"], "policy_question": False},
    {"id": "BEH-07", "category": "behavioral",
     "message": "Why is your return policy so strict? This is ridiculous.",
     "expected_keywords": ["understand", "sorry", "return"], "policy_question": False},
    {"id": "BEH-08", "category": "behavioral",
     "message": "I've been on hold for over an hour and nobody is helping me.",
     "expected_keywords": ["sorry", "apologize", "help"], "policy_question": False},
    {"id": "BEH-09", "category": "behavioral",
     "message": "This product is absolute garbage. I want my money back immediately.",
     "expected_keywords": ["sorry", "refund", "return"], "policy_question": False},
    {"id": "BEH-10", "category": "behavioral",
     "message": "Just tell me yes or no \u2014 can I return something I bought 35 days ago?",
     "expected_keywords": ["return", "days"], "policy_question": False},
    {"id": "BEH-11", "category": "behavioral",
     "message": "I don't understand your return policy at all. Can you explain it simply?",
     "expected_keywords": ["return", "days"], "policy_question": False},
    {"id": "BEH-12", "category": "behavioral",
     "message": "Is customer support available on weekends and holidays?",
     "expected_keywords": ["support", "help"], "policy_question": False},
    {"id": "BEH-13", "category": "behavioral",
     "message": "I am really stressed out about this order. Can you please just help me?",
     "expected_keywords": ["sorry", "help", "understand"], "policy_question": False},
    {"id": "BEH-14", "category": "behavioral",
     "message": "I want to return my item but I already threw away the original packaging.",
     "expected_keywords": ["return", "packaging"], "policy_question": False},
    {"id": "BEH-15", "category": "behavioral",
     "message": "My item arrived damaged but I still want to keep it. What can you do for me?",
     "expected_keywords": ["partial", "refund", "sorry"], "policy_question": False},
]

policy_cases    = [tc for tc in TEST_CASES if tc["category"] == "policy"]
behavioral_cases = [tc for tc in TEST_CASES if tc["category"] == "behavioral"]
print(f"Test cases loaded: {len(TEST_CASES)} total  "
      f"({len(policy_cases)} policy, {len(behavioral_cases)} behavioral)")

In [ ]:
# Cell 5: Load base Phi-4-mini-instruct in 4-bit quantization
# This is the RAW base model — no fine-tuning adapter is loaded.
# 4-bit NF4 quantization keeps VRAM usage under 8 GB on an A100.

MODEL_ID = "microsoft/Phi-4-mini-instruct"

print(f"Loading base model: {MODEL_ID}")
print("Quantization: 4-bit NF4 (BitsAndBytes)")
print("-" * 50)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

print(f"\nModel loaded successfully.")
print(f"Device map: {model.hf_device_map}")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"GPU memory — allocated: {allocated:.1f} GB  |  reserved: {reserved:.1f} GB")

In [ ]:
# Cell 6: Inference function
# Uses the Phi-4 chat template format and the Atlas system prompt.
# Returns (reply_text, latency_seconds).

SYSTEM_PROMPT = (
    "You are Atlas, a warm, direct, and helpful customer support representative "
    "for our e-commerce platform. You resolve shopping inquiries using verified "
    "system data. Keep replies concise and to the point."
)

PHI4_STOP_TOKENS = ["<|end|>", "<|user|>", "<|system|>"]


def generate_reply(message: str) -> tuple:
    """
    Generate a response from the base Phi-4-mini-instruct model.

    Args:
        message: The customer's message text.

    Returns:
        (reply_text: str, latency_s: float)
    """
    # Format using Phi-4 chat template
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{message}<|end|>\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = round(time.time() - t0, 2)

    # Decode only the newly generated tokens (exclude the prompt)
    new_tokens = output_ids[0][input_len:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Strip residual Phi-4 stop tokens if any survived special-token filtering
    for tok in PHI4_STOP_TOKENS:
        reply = reply.replace(tok, "").strip()

    return reply, latency


# Quick smoke test to confirm the model generates coherent output
print("Running smoke test...")
test_reply, test_lat = generate_reply("What is your return policy?")
print(f"Smoke test reply ({test_lat}s):\n{test_reply[:200]}")
print("\nSmoke test passed." if len(test_reply.split()) >= 5 else "WARNING: Smoke test reply suspiciously short.")

In [ ]:
# Cell 7: Run evaluation over all 40 test cases

print("=" * 70)
print("  BASE MODEL EVAL: microsoft/Phi-4-mini-instruct (no fine-tuning)")
print(f"  {len(TEST_CASES)} test cases  |  "
      f"{len(policy_cases)} policy  |  {len(behavioral_cases)} behavioral")
print("=" * 70)

all_results = []

for i, tc in enumerate(TEST_CASES, 1):
    # Generate model reply and measure latency
    reply, latency = generate_reply(tc["message"])

    # Score the reply on all 5 metrics
    compliance  = policy_compliance(reply)
    specificity = policy_specificity(reply, tc["policy_question"])
    relevance   = response_relevance(reply, tc["expected_keywords"])
    halluc      = hallucination_check(reply, tc["message"])
    fcr         = fcr_classify(reply)

    result = {
        "id":                 tc["id"],
        "category":           tc["category"],
        "message":            tc["message"],
        "reply":              reply,
        "reply_preview":      reply[:120] + "..." if len(reply) > 120 else reply,
        "compliance_score":   compliance["compliance_score"],
        "has_empathy":        compliance["has_empathy"],
        "has_next_step":      compliance["has_next_step"],
        "adequate_length":    compliance["adequate_length"],
        "professional_tone":  compliance["professional_tone"],
        "policy_specificity": specificity,
        "response_relevance": relevance,
        "hallucination":      halluc,
        "fcr":                fcr,
        "latency_s":          latency,
    }
    all_results.append(result)

    # Progress line
    h_sym = "H" if halluc else "-"
    e_sym = "E" if compliance["has_empathy"] else "-"
    print(
        f"  [{i:02d}/{len(TEST_CASES)}] {tc['id']:<8}  "
        f"comp={compliance['compliance_score']:.2f}  "
        f"spec={str(specificity):<4}  "
        f"rel={relevance:.2f}  "
        f"hal={h_sym}  emp={e_sym}  "
        f"fcr={fcr:<12}  {latency}s"
    )

print("\nEvaluation complete.")

In [ ]:
# Cell 8: Print summary results table

def avg(lst, key):
    """Mean of a numeric field, ignoring None values."""
    vals = [r[key] for r in lst if r[key] is not None]
    return round(mean(vals), 3) if vals else None

def pct(lst, key):
    """Mean of a boolean/binary field, ignoring None values."""
    vals = [r[key] for r in lst if r[key] is not None]
    return round(mean(int(v) for v in vals), 3) if vals else None


policy_res    = [r for r in all_results if r["category"] == "policy"]
behavioral_res = [r for r in all_results if r["category"] == "behavioral"]

# Aggregate across all 40 cases
comp_all      = avg(all_results, "compliance_score")
empathy_all   = pct(all_results, "has_empathy")
nextstep_all  = pct(all_results, "has_next_step")
comp_policy   = avg(policy_res,  "compliance_score")
spec_policy   = avg(policy_res,  "policy_specificity")
rel_all       = avg(all_results, "response_relevance")
halluc_rate   = pct(all_results, "hallucination")
latencies     = [r["latency_s"] for r in all_results if r["latency_s"] is not None]
lat_p50       = round(median(latencies), 2) if latencies else None
lat_p95       = round(float(np.percentile(latencies, 95)), 2) if latencies else None

total = len(all_results)
n_resolved  = sum(1 for r in all_results if r["fcr"] == "resolved")
n_followup  = sum(1 for r in all_results if r["fcr"] == "needs_followup")
n_escalated = sum(1 for r in all_results if r["fcr"] == "escalated")
fcr_str = (
    f"{100*n_resolved//total}% / "
    f"{100*n_followup//total}% / "
    f"{100*n_escalated//total}%"
)

# ── Print formatted table ─────────────────────────────────────────────────────
W = 49
print("\n" + "=" * W)
print(f"  {'Metric':<34} {'Base Phi-4-mini':>12}")
print("\u2500" * W)
print(f"  {'Compliance Rate (all 40)':<34} {comp_all:>12.3f}")
print(f"  {'  Empathy Rate':<34} {empathy_all:>12.3f}")
print(f"  {'  Next-Step Rate':<34} {nextstep_all:>12.3f}")
print(f"  {'Compliance Rate (policy only)':<34} {comp_policy:>12.3f}")
print(f"  {'Policy Specificity Score':<34} {spec_policy:>12.3f}")
print(f"  {'Response Relevance (all)':<34} {rel_all:>12.3f}")
print(f"  {'Hallucination Rate':<34} {halluc_rate:>12.3f}")
print(f"  {'FCR: resolved / follow-up / esc':<34} {fcr_str:>12}")
print(f"  {'Latency p50 (s)':<34} {lat_p50:>12.2f}")
print(f"  {'Latency p95 (s)':<34} {lat_p95:>12.2f}")
print("=" * W)

In [ ]:
# Cell 9: Save raw results to base_model_results.json
# Saved to the same directory as this notebook (eval/).

output = {
    "model": MODEL_ID,
    "quantization": "4-bit NF4 (BitsAndBytes)",
    "n_cases": len(all_results),
    "summary": {
        "compliance_rate_all":    comp_all,
        "empathy_rate":           empathy_all,
        "nextstep_rate":          nextstep_all,
        "compliance_rate_policy": comp_policy,
        "policy_specificity":     spec_policy,
        "response_relevance":     rel_all,
        "hallucination_rate":     halluc_rate,
        "fcr_resolved_pct":       round(n_resolved  / total, 3),
        "fcr_followup_pct":       round(n_followup  / total, 3),
        "fcr_escalated_pct":      round(n_escalated / total, 3),
        "latency_p50_s":          lat_p50,
        "latency_p95_s":          lat_p95,
    },
    "results": all_results,
}

# Resolve output path relative to this notebook file
try:
    notebook_dir = Path(__file__).parent
except NameError:
    # Running inside Jupyter — __file__ is not defined
    notebook_dir = Path(".").resolve()

out_path = notebook_dir / "base_model_results.json"
out_path.write_text(json.dumps(output, indent=2, default=str))
print(f"Results saved to: {out_path}")
print(f"  {len(all_results)} records  |  summary keys: {list(output['summary'].keys())}")

## Side-by-Side Example Comparisons

The three examples below highlight where the base model **succeeds** (high compliance) and where it **struggles** (low policy specificity). 
These gaps are expected to narrow significantly with fine-tuning and RAG grounding.

In [ ]:
# Cell 10: Show 3 illustrative example responses with per-metric scores

def fmt_score(val, width=5):
    """Format a score value for display."""
    if val is None:
        return "N/A  "
    if isinstance(val, bool):
        return "Yes  " if val else "No   "
    return f"{val:.2f} "


def print_example(result, idx):
    """Pretty-print a single result with all metric scores."""
    sep = "-" * 68
    print(f"\n{'=' * 68}")
    print(f"  Example {idx}: [{result['id']}] ({result['category'].upper()})")
    print(sep)
    print(f"  Customer message:")
    print(f"    {result['message']}")
    print(sep)
    print(f"  Base model reply:")
    # Wrap reply at ~64 chars for readability
    words = result['reply'].split()
    line, lines = [], []
    for w in words:
        line.append(w)
        if len(" ".join(line)) > 64:
            lines.append("    " + " ".join(line[:-1]))
            line = [w]
    if line:
        lines.append("    " + " ".join(line))
    print("\n".join(lines))
    print(sep)
    print(f"  Scores:")
    print(f"    Compliance        : {fmt_score(result['compliance_score'])}  "
          f"(empathy={result['has_empathy']}, next_step={result['has_next_step']})")
    print(f"    Policy Specificity: {fmt_score(result['policy_specificity'])}")
    print(f"    Response Relevance: {fmt_score(result['response_relevance'])}")
    print(f"    Hallucination     : {'YES' if result['hallucination'] else 'No'}")
    print(f"    FCR               : {result['fcr']}")
    print(f"    Latency           : {result['latency_s']}s")


# Select 3 informative examples:
#   1. Best compliance score (shows model strengths)
#   2. Worst policy specificity among policy questions (shows RAG gap)
#   3. A behavioral case with highest relevance

# Example 1: highest compliance overall
ex1 = max(all_results, key=lambda r: r["compliance_score"])

# Example 2: policy case with lowest specificity (biggest gap vs RAG)
policy_res_sorted = sorted(
    policy_res,
    key=lambda r: r["policy_specificity"] if r["policy_specificity"] is not None else 999
)
ex2 = policy_res_sorted[0]

# Example 3: behavioral case with highest response relevance
ex3 = max(behavioral_res, key=lambda r: r["response_relevance"])

print_example(ex1, 1)
print_example(ex2, 2)
print_example(ex3, 3)

print(f"\n{'=' * 68}")
print("  Note: Policy specificity is expected to be low without RAG.")
print("  The fine-tuned + RAG system retrieves exact policy details")
print("  (e.g., '30 days', '3-5 business days', 'pre-paid return label')")
print("  that the base model must guess from pre-training knowledge.")
print("=" * 68)